### Changed Forward pass
Needed to record both spiking data seperatly. This will cause errors, because the output now contains 2 seperate recordings. mb

In [ ]:
def forward_pass(net, spike_data):
    """Run the network over time on a pre-encoded spike train.

    spike_data: [num_steps, batch, 1, 28, 28] (from spikegen.rate)
    Returns stacked hidden and stacked output spikes: [num_steps, batch, num_outputs]
    """
    spk_rec_output = []
    spk_rec_hidden = []
    t_hidden_history = []
    t_output_history = []
    utils.reset(net)                       # clear membrane states of every Leaky

    for step in range(spike_data.size(0)):
        x_t = spike_data[step]
        x_t = x_t.view(x_t.size(0), -1)   # shape: [batch, 784]

        x_hidden = net[0](x_t)
        spk_hidden = net[1](x_hidden)
        spk_rec_hidden.append(spk_hidden)  # Save hidden
        
        x_out = net[2](spk_hidden)
        spk_out, _ = net[3](x_out)
        spk_rec_output.append(spk_out)  # Save output

        # Log thresholds of hidden layer neurons
        t_hidden_history.append(net[1].threshold.clone().detach().cpu())
        # Log thresholds of output layer neurons
        t_output_history.append(net[3].threshold.clone().detach().cpu())

    return torch.stack(spk_rec_output), torch.stack(spk_rec_hidden)

### function to test a trained model, and record relevant data.

In [ ]:
def test_model(net, test_loader, num_steps=25, device=None):
    """
    Test a trained model on the test set and collect all necessary data for evaluation.
    Returns a dictionary with overall statistics.
    """
    net.eval()
    
    all_output_spikes = []
    all_hidden_spikes = []
    all_targets = []
    
    print(f"\nTesting model on test set...")
    print("Processing batches...", end=" ")
    
    with torch.no_grad():
        for batch_idx, (data, targets) in enumerate(test_loader):
            data = data.to(device)
            
            # Encode to spikes
            spike_data = spikegen.rate(data, num_steps=num_steps)
            
            # Forward pass (returns both hidden and output spikes)
            spk_out, spk_hidden = forward_pass(net, spike_data)
            
            # Store spike data
            all_output_spikes.append(spk_out.cpu())
            all_hidden_spikes.append(spk_hidden.cpu())
            all_targets.append(targets)
            
            if (batch_idx + 1) % 10 == 0:
                print(f"{batch_idx + 1}...", end=" ")
    
    print("Done!")
    
    # === AGGREGATE ALL DATA ===
    
    # Concatenate along batch dimension and permute to [samples, timesteps, neurons]
    output_spikes = torch.cat(all_output_spikes, dim=1).permute(1, 0, 2)
    hidden_spikes = torch.cat(all_hidden_spikes, dim=1).permute(1, 0, 2)
    targets = torch.cat(all_targets)
    
    # === GET PREDICTIONS ===
    
    # Sum spikes over time and get predicted class
    output_spikes_summed = output_spikes.sum(dim=1)
    predictions = output_spikes_summed.argmax(dim=1)
    
    # === COMPUTE METRICS ===
    
    from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, confusion_matrix
    
    targets_np = targets.numpy()
    predictions_np = predictions.numpy()
    
    # Overall metrics (macro average - simple average across classes)
    accuracy = accuracy_score(targets_np, predictions_np)
    precision = precision_score(targets_np, predictions_np, average='macro')
    recall = recall_score(targets_np, predictions_np, average='macro')
    f1 = f1_score(targets_np, predictions_np, average='macro')
    
    # Confusion matrix (for visualization if needed)
    cm = confusion_matrix(targets_np, predictions_np, labels=range(10))
    
    # === FIRING RATES ===
    
    hidden_firing_rates = hidden_spikes.mean(dim=(0, 1))
    output_firing_rates = output_spikes.mean(dim=(0, 1))
    
    # === SPARSITY ===
    
    hidden_sparsity = (hidden_spikes == 0).float().mean().item() * 100
    output_sparsity = (output_spikes == 0).float().mean().item() * 100
    
    hidden_dead = (hidden_firing_rates == 0).sum().item()
    hidden_saturated = (hidden_firing_rates > 0.5).sum().item()
    
    # === PACKAGE RESULTS ===
    
    results = {
        # Raw spike data
        'output_spikes': output_spikes,
        'hidden_spikes': hidden_spikes,
        'targets': targets,
        'predictions': predictions,
        'confusion_matrix': cm,
        
        # Performance metrics (overall)
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        
        # Firing rates
        'hidden_firing_rates': hidden_firing_rates,
        'output_firing_rates': output_firing_rates,
        
        # Sparsity metrics
        'hidden_sparsity': hidden_sparsity,
        'output_sparsity': output_sparsity,
        'hidden_dead': hidden_dead,
        'hidden_saturated': hidden_saturated,
    }
    
    # === PRINT SUMMARY ===
    print("\n" + "="*70)
    print("TEST RESULTS SUMMARY")
    print("="*70)
    print(f"Accuracy:                    {accuracy*100:.2f}%")
    print(f"Precision:                   {precision*100:.2f}%")
    print(f"Recall:                      {recall*100:.2f}%")
    print(f"F1-Score:                    {f1*100:.2f}%")
    print(f"\nHidden Sparsity:             {hidden_sparsity:.2f}%")
    print(f"Output Sparsity:             {output_sparsity:.2f}%")
    print(f"Dead Neurons (hidden):       {hidden_dead}")
    print(f"Saturated Neurons (hidden):  {hidden_saturated}")
    print("="*70 + "\n")
    
    return results

### Plotting a simple CM

In [ ]:
def plot_confusion_matrix(test_results, name="Network"):
    """
    Plot confusion matrix from test results.
    Shows which digits get confused with which.
    """
    cm = test_results['confusion_matrix']
    targets = test_results['targets'].numpy()
    predictions = test_results['predictions'].numpy()
    
    from sklearn.metrics import accuracy_score
    accuracy = accuracy_score(targets, predictions)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create heatmap
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    
    # Add text annotations
    for i in range(10):
        for j in range(10):
            text = ax.text(j, i, cm[i, j],
                          ha="center", va="center",
                          color="white" if cm[i, j] > cm.max() / 2 else "black",
                          fontsize=11, fontweight='bold')
    
    # Labels and title
    ax.set_xlabel('Predicted Digit', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Digit', fontsize=12, fontweight='bold')
    ax.set_title(f'Confusion Matrix: {name}\nAccuracy: {accuracy*100:.2f}%',
                fontsize=13, fontweight='bold')
    
    # Ticks
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, label='Number of Samples')
    
    # Grid
    ax.grid(False)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n" + "="*70)
    print("CONFUSION MATRIX ANALYSIS")
    print("="*70)
    print(f"Total Samples: {cm.sum()}")
    print(f"Correct Predictions: {cm.diagonal().sum()}")
    print(f"Incorrect Predictions: {cm.sum() - cm.diagonal().sum()}")
    print(f"Overall Accuracy: {accuracy*100:.2f}%")
    print("="*70 + "\n")

### Rasterplot with firing rates joined

In [ ]:
def plot_sparsity_raster_with_hip(test_results, hip_target_rates=None, name="Network", sample_idx=0, num_hidden_show=150):
    """
    Plot raster diagrams with marginal firing rate histograms and HIP target lines.
    Shows activity for a single sample over timesteps.
    
    test_results: Dictionary from test_model()
    hip_target_rates: list [hidden_target, output_target] (e.g., [0.05, 0.05])
    sample_idx: Which sample to visualize (default: 0, first sample)
    """
    
    output_spikes = test_results['output_spikes']  # [samples, timesteps, 10]
    hidden_spikes = test_results['hidden_spikes']  # [samples, timesteps, num_hidden]
    
    # Extract single sample
    hidden_sample = hidden_spikes[sample_idx, :, :]  # [timesteps, num_hidden]
    output_sample = output_spikes[sample_idx, :, :]  # [timesteps, num_outputs]
    
    # ============ HIDDEN LAYER JOINT PLOT ============
    print(f"Generating Hidden Layer visualization (Sample {sample_idx})...")
    
    fig_h = plt.figure(figsize=(14, 10), constrained_layout=True)
    gs_h = fig_h.add_gridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                             hspace=0.05, wspace=0.05)
    
    num_neurons_show = min(num_hidden_show, hidden_sample.shape[1])
    num_timesteps = hidden_sample.shape[0]
    
    # Prepare data for raster (single sample)
    sample_hidden_subset = hidden_sample[:, :num_neurons_show]
    
    # Raster plot (bottom-left)
    ax_raster_h = fig_h.add_subplot(gs_h[1, 0])
    
    spike_times, neuron_ids = torch.where(sample_hidden_subset > 0)
    
    ax_raster_h.scatter(spike_times.numpy(), neuron_ids.numpy(),
                       marker='|', s=20, alpha=0.7, color='darkblue')
    ax_raster_h.set_xlabel('Timestep', fontsize=11, fontweight='bold')
    ax_raster_h.set_ylabel('Neuron ID', fontsize=11, fontweight='bold')
    ax_raster_h.set_title(f'Raster Plot (First {num_neurons_show} neurons)', 
                         fontsize=12, fontweight='bold')
    ax_raster_h.grid(True, alpha=0.1)
    ax_raster_h.set_xlim([-0.5, num_timesteps - 0.5])
    ax_raster_h.set_xticks(range(0, num_timesteps, 5))
    
    # Firing rate over timesteps (top-left)
    ax_activity_h = fig_h.add_subplot(gs_h[0, 0], sharex=ax_raster_h)
    
    # Firing rate for each timestep (across neurons)
    firing_rate_per_timestep_h = sample_hidden_subset.mean(dim=1) * 100
    
    ax_activity_h.plot(range(num_timesteps), firing_rate_per_timestep_h.numpy(),
                      color='steelblue', linewidth=2.5, marker='o', markersize=4)
    ax_activity_h.fill_between(range(num_timesteps), firing_rate_per_timestep_h.numpy(),
                              alpha=0.3, color='steelblue')
    
    # Add HIP target line
    if hip_target_rates and len(hip_target_rates) > 0:
        target = hip_target_rates[0] * 100
        ax_activity_h.axhline(target, color='red', linestyle='--', linewidth=2.5,
                             label=f'HIP Target: {target:.1f}%', alpha=0.8)
        ax_activity_h.legend(loc='upper right', fontsize=10)
    
    ax_activity_h.set_ylabel('Firing Rate (%)', fontsize=11, fontweight='bold')
    ax_activity_h.set_title('Firing Rate Over Timesteps', fontsize=11, fontweight='bold')
    ax_activity_h.grid(True, alpha=0.2, axis='y')
    ax_activity_h.tick_params(labelbottom=False)
    ax_activity_h.set_ylim([0, max(firing_rate_per_timestep_h) * 1.2])
    
    # Per-neuron firing rates (right side) - average across timesteps
    ax_firing_h = fig_h.add_subplot(gs_h[1, 1], sharey=ax_raster_h)
    
    neuron_firing_h = sample_hidden_subset.mean(dim=0) * 100
    
    ax_firing_h.barh(range(num_neurons_show), neuron_firing_h.numpy(),
                    color='steelblue', alpha=0.6, height=0.8)
    
    # Add HIP target line
    if hip_target_rates and len(hip_target_rates) > 0:
        target = hip_target_rates[0] * 100
        ax_firing_h.axvline(target, color='red', linestyle='--', linewidth=2.5,
                           label=f'Target: {target:.1f}%', alpha=0.8)
        ax_firing_h.legend(loc='lower right', fontsize=9)
    
    ax_firing_h.set_xlabel('Firing Rate (%)', fontsize=11, fontweight='bold')
    ax_firing_h.tick_params(labelleft=False)
    ax_firing_h.grid(True, alpha=0.2, axis='x')
    ax_firing_h.set_xlim([0, max(neuron_firing_h) * 1.2])
    
    plt.show()
    
    # ============ OUTPUT LAYER JOINT PLOT ============
    print("Generating Output Layer visualization...")
    
    fig_o = plt.figure(figsize=(12, 8), constrained_layout=True)
    gs_o = fig_o.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[1, 4],
                             hspace=0.05, wspace=0.05)
    
    num_timesteps_o = output_sample.shape[0]
    
    # Raster plot (bottom-left)
    ax_raster_o = fig_o.add_subplot(gs_o[1, 0])
    
    spike_times_o, neuron_ids_o = torch.where(output_sample > 0)
    
    ax_raster_o.scatter(spike_times_o.numpy(), neuron_ids_o.numpy(),
                       marker='|', s=100, alpha=0.8, color='green')
    ax_raster_o.set_xlabel('Timestep', fontsize=11, fontweight='bold')
    ax_raster_o.set_ylabel('Output Neuron (Digit)', fontsize=11, fontweight='bold')
    ax_raster_o.set_title('Raster Plot', fontsize=12, fontweight='bold')
    ax_raster_o.set_yticks(range(10))
    ax_raster_o.grid(True, alpha=0.1)
    ax_raster_o.set_xlim([-0.5, num_timesteps_o - 0.5])
    ax_raster_o.set_xticks(range(0, num_timesteps_o, 5))
    
    # Firing rate over timesteps (top-left)
    ax_activity_o = fig_o.add_subplot(gs_o[0, 0], sharex=ax_raster_o)
    
    # Firing rate for each timestep (across output neurons)
    firing_rate_per_timestep_o = output_sample.mean(dim=1) * 100
    
    ax_activity_o.plot(range(num_timesteps_o), firing_rate_per_timestep_o.numpy(),
                      color='green', linewidth=2.5, marker='o', markersize=5)
    ax_activity_o.fill_between(range(num_timesteps_o), firing_rate_per_timestep_o.numpy(),
                              alpha=0.3, color='green')
    
    # Add HIP target line
    if hip_target_rates and len(hip_target_rates) > 1:
        target = hip_target_rates[1] * 100
        ax_activity_o.axhline(target, color='red', linestyle='--', linewidth=2.5,
                             label=f'HIP Target: {target:.1f}%', alpha=0.8)
        ax_activity_o.legend(loc='upper right', fontsize=10)
    
    ax_activity_o.set_ylabel('Firing Rate (%)', fontsize=11, fontweight='bold')
    ax_activity_o.set_title('Firing Rate Over Timesteps', fontsize=11, fontweight='bold')
    ax_activity_o.grid(True, alpha=0.2, axis='y')
    ax_activity_o.tick_params(labelbottom=False)
    ax_activity_o.set_ylim([0, max(firing_rate_per_timestep_o) * 1.2])
    
    # Per-neuron firing rates (right side) - average across timesteps
    ax_firing_o = fig_o.add_subplot(gs_o[1, 1], sharey=ax_raster_o)
    
    neuron_firing_o = output_sample.mean(dim=0) * 100
    
    colors_output = ['green' if rate > 0 else 'lightgray' for rate in neuron_firing_o]
    ax_firing_o.barh(range(10), neuron_firing_o.numpy(),
                    color=colors_output, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add HIP target line
    if hip_target_rates and len(hip_target_rates) > 1:
        target = hip_target_rates[1] * 100
        ax_firing_o.axvline(target, color='red', linestyle='--', linewidth=2.5,
                           label=f'Target: {target:.1f}%', alpha=0.8)
        ax_firing_o.legend(loc='lower right', fontsize=9)
    
    ax_firing_o.set_xlabel('Firing Rate (%)', fontsize=11, fontweight='bold')
    ax_firing_o.tick_params(labelleft=False)
    ax_firing_o.grid(True, alpha=0.2, axis='x')
    ax_firing_o.set_xlim([0, max(neuron_firing_o) * 1.2])
    
    # Add value labels on output firing rates
    for i, rate in enumerate(neuron_firing_o):
        ax_firing_o.text(rate + 0.5, i, f'{rate:.1f}%', va='center', fontsize=9)
    
    plt.show()
    
    # === HIP ADHERENCE ANALYSIS ===
    if hip_target_rates:
        print("\n" + "="*70)
        print(f"HIP TARGET ADHERENCE (Sample {sample_idx})")
        print("="*70)
        
        # Hidden layer
        target_h = hip_target_rates[0]
        firing_h = hidden_sample.mean(dim=0)
        within_h = ((firing_h > target_h * 0.8) & (firing_h < target_h * 1.2)).float().mean().item() * 100
        above_h = (firing_h > target_h * 1.2).float().mean().item() * 100
        below_h = (firing_h < target_h * 0.8).float().mean().item() * 100
        
        print(f"\nHIDDEN LAYER (Target: {target_h*100:.1f}%):")
        print(f"  Within ±20%:  {within_h:.1f}%")
        print(f"  Above target: {above_h:.1f}%")
        print(f"  Below target: {below_h:.1f}%")
        print(f"  Mean firing:  {firing_h.mean()*100:.2f}%")
        
        # Output layer
        if len(hip_target_rates) > 1:
            target_o = hip_target_rates[1]
            firing_o = output_sample.mean(dim=0)
            within_o = ((firing_o > target_o * 0.8) & (firing_o < target_o * 1.2)).float().mean().item() * 100
            above_o = (firing_o > target_o * 1.2).float().mean().item() * 100
            below_o = (firing_o < target_o * 0.8).float().mean().item() * 100
            
            print(f"\nOUTPUT LAYER (Target: {target_o*100:.1f}%):")
            print(f"  Within ±20%:  {within_o:.1f}%")
            print(f"  Above target: {above_o:.1f}%")
            print(f"  Below target: {below_o:.1f}%")
            print(f"  Mean firing:  {firing_o.mean()*100:.2f}%")
        
        print("="*70 + "\n")

## Plotting thresholds

### Lineplot

In [ ]:
def plot_threshold_evolution_lines(threshold_log_hidden, threshold_log_output, name="Network", num_neurons_sample=10):
    """
    Alternative visualization: plot threshold changes as lines for selected neurons.
    """
    
    thresholds_h = torch.stack(threshold_log_hidden).numpy()
    thresholds_o = torch.stack(threshold_log_output).numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    
    # Hidden layer - sample some neurons
    ax = axes[0]
    neurons_to_plot = min(num_neurons_sample, thresholds_h.shape[1])
    
    for i in range(neurons_to_plot):
        ax.plot(thresholds_h[:, i], alpha=0.7, linewidth=1.5)
    
    ax.set_xlabel('Training Iteration', fontsize=11, fontweight='bold')
    ax.set_ylabel('Threshold Value', fontsize=11, fontweight='bold')
    ax.set_title(f'Hidden Layer Thresholds (Sample of {neurons_to_plot} neurons)',
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Output layer - all 10 neurons
    ax = axes[1]
    
    for i in range(10):
        ax.plot(thresholds_o[:, i], label=f'Digit {i}', linewidth=2, marker='o', markersize=3)
    
    ax.set_xlabel('Training Iteration', fontsize=11, fontweight='bold')
    ax.set_ylabel('Threshold Value', fontsize=11, fontweight='bold')
    ax.set_title('Output Layer Thresholds (All Classes)',
                fontsize=12, fontweight='bold')
    ax.legend(loc='best', ncol=2, fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.show()

# Usage
plot_threshold_evolution_lines(threshold_log_hidden, threshold_log_output, num_neurons_sample=20)

### heatmap

In [ ]:
def plot_threshold_evolution(threshold_log_hidden, threshold_log_output, name="Network"):
    """
    Visualize how neuron thresholds change over training iterations.
    Uses heatmap: white=low threshold (easy to fire), black=high threshold (hard to fire)
    
    threshold_log_hidden: list of tensors from train_model() [num_iterations, num_hidden]
    threshold_log_output: list of tensors from train_model() [num_iterations, num_outputs]
    """
    
    # Stack into single tensors: [iterations, neurons]
    thresholds_h = torch.stack(threshold_log_hidden).numpy()  # [iterations, num_hidden]
    thresholds_o = torch.stack(threshold_log_output).numpy()  # [iterations, num_outputs]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    
    # ============ HIDDEN LAYER THRESHOLDS ============
    ax = axes[0]
    
    # Show subset of neurons if there are too many
    num_neurons_show = min(200, thresholds_h.shape[1])
    thresholds_h_subset = thresholds_h[:, :num_neurons_show]
    
    # Invert colormap: white=low, black=high (use 'gray_r' for reversed)
    im_h = ax.imshow(thresholds_h_subset.T, aspect='auto', cmap='gray_r', interpolation='nearest')
    
    ax.set_xlabel('Training Iteration', fontsize=12, fontweight='bold')
    ax.set_ylabel('Neuron ID', fontsize=12, fontweight='bold')
    ax.set_title(f'Hidden Layer Threshold Evolution (First {num_neurons_show} neurons)',
                fontsize=13, fontweight='bold')
    
    cbar_h = plt.colorbar(im_h, ax=ax, label='Threshold Value')
    
    # Add grid
    ax.grid(False)
    
    # ============ OUTPUT LAYER THRESHOLDS ============
    ax = axes[1]
    
    im_o = ax.imshow(thresholds_o.T, aspect='auto', cmap='gray_r', interpolation='nearest')
    
    ax.set_xlabel('Training Iteration', fontsize=12, fontweight='bold')
    ax.set_ylabel('Output Neuron (Digit)', fontsize=12, fontweight='bold')
    ax.set_title('Output Layer Threshold Evolution',
                fontsize=13, fontweight='bold')
    ax.set_yticks(range(10))
    
    cbar_o = plt.colorbar(im_o, ax=ax, label='Threshold Value')
    
    # Add grid
    ax.grid(False)
    
    plt.show()
    
    # === PRINT STATISTICS ===
    print("\n" + "="*70)
    print("THRESHOLD EVOLUTION STATISTICS")
    print("="*70)
    
    print(f"\nHIDDEN LAYER (showing first {num_neurons_show} neurons):")
    print(f"  Initial thresholds: min={thresholds_h_subset[0].min():.4f}, "
          f"max={thresholds_h_subset[0].max():.4f}, "
          f"mean={thresholds_h_subset[0].mean():.4f}")
    print(f"  Final thresholds:   min={thresholds_h_subset[-1].min():.4f}, "
          f"max={thresholds_h_subset[-1].max():.4f}, "
          f"mean={thresholds_h_subset[-1].mean():.4f}")
    print(f"  Total change:       min={thresholds_h_subset[-1].min() - thresholds_h_subset[0].min():.4f}, "
          f"max={thresholds_h_subset[-1].max() - thresholds_h_subset[0].max():.4f}, "
          f"mean={thresholds_h_subset[-1].mean() - thresholds_h_subset[0].mean():.4f}")
    
    print(f"\nOUTPUT LAYER:")
    print(f"  Initial thresholds: min={thresholds_o[0].min():.4f}, "
          f"max={thresholds_o[0].max():.4f}, "
          f"mean={thresholds_o[0].mean():.4f}")
    print(f"  Final thresholds:   min={thresholds_o[-1].min():.4f}, "
          f"max={thresholds_o[-1].max():.4f}, "
          f"mean={thresholds_o[-1].mean():.4f}")
    print(f"  Total change:       min={thresholds_o[-1].min() - thresholds_o[0].min():.4f}, "
          f"max={thresholds_o[-1].max() - thresholds_o[0].max():.4f}, "
          f"mean={thresholds_o[-1].mean() - thresholds_o[0].mean():.4f}")
    
    print("="*70 + "\n")
    
    return thresholds_h, thresholds_o


# Usage (after training)
thresholds_h, thresholds_o = plot_threshold_evolution(
    threshold_log_hidden, 
    threshold_log_output,
    name=f"Network (HIP={HIP})"
)

## Example

In [ ]:
# Usage (after training)
thresholds_h, thresholds_o = plot_threshold_evolution(
    threshold_log_hidden, 
    threshold_log_output,
    name=f"Network (HIP={HIP})")

plot_threshold_evolution_lines(threshold_log_hidden, threshold_log_output, num_neurons_sample=20)

test_results = test_model(net, test_loader, num_steps=num_steps, device=device)

plot_confusion_matrix(test_results, name=f"Network (HIP={HIP})")

if HIP:
    hip_targets = target_rates  # From your training parameters
else:
    hip_targets = None

plot_sparsity_raster_with_hip(test_results, 
                              hip_target_rates=hip_targets,
                              name=f"Network (HIP={HIP})",
                              sample_idx=42,
                              num_hidden_show=500)

hidden_firing_avg = test_results['hidden_firing_rates'].mean()
print(f"Average hidden firing rate (all samples): {hidden_firing_avg*100:.2f}%")